# rules
> Static AST checks that catch common agent kernel mistakes before execution

In [ ]:
#| default_exp rules

The router checks each `exec` cell against these rules before forwarding it. Advisory rules attach a note to the reply; blocking rules stop the cell before it reaches the gateway. Rules inspect only the cell text, not which kernel runs it, so each check targets Python forms that no other kernel language produces.

In [ ]:
import ast,re,tokenize
from IPython.core.inputtransformer2 import TransformerManager
from io import StringIO
from fastcore.basics import store_attr

In [ ]:
from fastcore.test import *

## Rules and `scan`

`Rule` pairs a predicate `fn(tree, src)` with the `note` returned when it fires. Set `block=True` to reject the cell instead of forwarding it, and `raw=True` to check the text as typed rather than after IPython's input transform. `scan` transforms the cell with IPython's `TransformerManager` so magic lines parse as Python, returning every rule in `RULES` that fires (or `[]` when the text does not parse as Python).

In [ ]:
class Rule:
    "A named check: `fn(tree, src)` is truthy when `note` applies; `block` stops the cell, and `raw` checks the source as typed rather than as IPython transforms it"
    def __init__(self, name, note, fn, block=False, raw=False): store_attr()

RULES = []
_tm = TransformerManager()

def _transform(src):
    "IPython-transform `src` so magics and `!` escapes parse; the raw text comes back if transformation fails"
    try: return _tm.transform_cell(src)
    except Exception: return src

def scan(src):
    "The rules that fire on cell `src`, parsed as IPython would parse it"
    tsrc = _transform(src)
    try: tree = ast.parse(tsrc)
    except SyntaxError: return []
    return [r for r in RULES if r.fn(tree, src if r.raw else tsrc)]

Two small AST helpers walk call nodes and resolve plain function names:

In [ ]:
def _calls(tree):
    for n in ast.walk(tree):
        if isinstance(n, ast.Call): yield n

def _callee(c): return c.func.id if isinstance(c.func, ast.Name) else None

The lessons below use `fires(src, name)` to test whether a named rule triggers on a cell. Cells that fail Python parsing (such as non-Python kernel languages) return no rules:

In [ ]:
def fires(src, name): return any(r.name == name for r in scan(src))
test_eq(scan('def broken('), [])

## File reads and edits

Displayed `read_text()` or `open().read()` calls on text files dump raw unnumbered text into context instead of using `open_doc`, `lnhashview_file`, or `view_file`. Assigned or parser-bound reads, variable paths whose extension is unknown, and structured data files (`.json`, `.jsonl`, `.csv`, `.log`) stay quiet.

In [ ]:
_DATA_EXTS = ('.json','.jsonl','.ndjson','.csv','.tsv','.log')

def _textpath(node):
    "A string constant under `node` recognisably names a non-data file: only then is a view-note earned"
    return any(isinstance(n, ast.Constant) and isinstance(n.value, str) and '.' in n.value
        and not n.value.lower().endswith(_DATA_EXTS) for n in ast.walk(node))

def _is_read(c):
    "A read_text()/open().read() call on a recognisably-named non-data file"
    if not (isinstance(c, ast.Call) and isinstance(c.func, ast.Attribute)): return False
    if c.func.attr == 'read_text': return _textpath(c.func.value)
    return c.func.attr == 'read' and isinstance(c.func.value, ast.Call) and _callee(c.func.value) == 'open' and _textpath(c.func.value)

def _read_file(tree, src):
    "Only a displayed read (bare expression or print) earns the note: a parser-bound or assigned read never enters context"
    for n in ast.walk(tree):
        if isinstance(n, ast.Expr):
            v = n.value
            if _is_read(v): return True
            if isinstance(v, ast.Call) and _callee(v) == 'print' and any(_is_read(a) for a in v.args): return True

RULES.append(Rule('read_file', 'Read files with open_doc (large), lnhashview_file (for editing) or view_file(nums=False).', _read_file))

`read_file` fires on bare or printed reads of text paths, including after magic lines, and stays quiet on variable paths, data files, assignments, and parser-bound reads:

In [ ]:
hit = ("Path('core.py').read_text()", "open('notes.txt').read()", "(d/'core.py').read_text()",
    "print(Path('a.py').read_text())", "%nbrun ab12\nPath('f.py').read_text()")
quiet = ('lnhashview_file(p)', 'p.read_text()', 'open(p).read()', "open(p, 'r').read()", "Path('trace.jsonl').read_text()",
    "open(d/'rows.csv').read()", "yaml.safe_load(Path('wf.yaml').read_text())", "src = Path('a.py').read_text()")
assert all(fires(s, 'read_file') for s in hit)
assert not any(fires(s, 'read_file') for s in quiet)
[(r.name, r.block, r.note) for r in scan(hit[0])]

Large `file_replace_lines` or `cell_replace_lines` payloads (six or more newlines) and single-cell `cell_str_replace` calls with a literal cell ID point back to `%%exhash`. Batch `cell_str_replace` calls targeting `'all'`, a list of IDs, or a variable stay quiet.

In [ ]:
def _big_replace(tree, src):
    for c in _calls(tree):
        if _callee(c) in ('file_replace_lines','cell_replace_lines'):
            for k in c.keywords:
                if k.arg == 'new_content' and isinstance(k.value, ast.Constant) and str(k.value.value).count('\n') >= 6: return True

def _cell_str_replace(tree, src):
    "Only the obvious single-cell form (a literal id that isn't 'all'): batch replaces over many cells are sanctioned"
    for c in _calls(tree):
        if _callee(c) != 'cell_str_replace': continue
        cid = c.args[0] if c.args else next((k.value for k in c.keywords if k.arg == 'id'), None)
        if isinstance(cid, ast.Constant) and isinstance(cid.value, str) and cid.value != 'all': return True

RULES.append(Rule('big_replace', 'Replace a whole cell or file with %%exhash <path> [<cell_id>] % c; an inner region with a range-c address.', _big_replace))
RULES.append(Rule('cell_str_replace', 'Edit notebook cells with %%exhash <path> <cell_id>.', _cell_str_replace))

Single-line replacements and batch `cell_str_replace` forms stay quiet while multi-line replacements and single-cell literal IDs fire:

In [ ]:
big = 'x = 1\n' * 9
assert fires(f'file_replace_lines(p, new_content={big!r})', 'big_replace')
assert fires(f'cell_replace_lines(cid, new_content={big!r})', 'big_replace')
assert not fires("file_replace_lines(p, new_content='x = 1')", 'big_replace')
assert fires("cell_str_replace('ab12', 'a', 'b', fname=p)", 'cell_str_replace')
batch = ("cell_str_replace(['ab12','cd34'], 'a', 'b')", "cell_str_replace('all', 'a', 'b')", "cell_str_replace(cids, 'a', 'b')")
assert not any(fires(s, 'cell_str_replace') for s in batch)

## Raw strings and exhash

Non-raw triple-quoted strings containing backslashes risk silent escape corruption, and calling `lnhash` or `line_hash` computes addresses instead of copying them from a fresh view.

In [ ]:
def _rawstr(tree, src):
    try: toks = list(tokenize.generate_tokens(StringIO(src).readline))
    except tokenize.TokenizeError: return
    for t in toks:
        if t.type == tokenize.STRING:
            m = re.match(r"""([A-Za-z]*)('''|\"\"\")""", t.string)
            if m and 'r' not in m[1].lower() and '\\' in t.string: return True

def _hashcalc(tree, src): return any(_callee(c) in ('lnhash','line_hash') for c in _calls(tree))

RULES.append(Rule('rawstr', 'Write non-trivial strings as r""" raw strings; %%exhash text needs no escaping at all.', _rawstr))
RULES.append(Rule('hashcalc', 'exhash addresses come only from a fresh lnhashview; never compute them.', _hashcalc))

Raw triple quotes and backslash-free triple quotes pass `rawstr`, while both hash helpers fire `hashcalc`:

In [ ]:
assert fires('s = """a\\nb"""', 'rawstr')
assert not fires('s = r"""a\\nb"""', 'rawstr')
assert not fires('s = """plain text"""', 'rawstr')
assert fires('addr = lnhash(3, line)', 'hashcalc') and fires('line_hash(s)', 'hashcalc')

Single-command `exhash`, `file_exhash`, and `cell_exhash` calls with long or quote-heavy `a`/`i`/`c` tuple payloads belong in a `%%exhash` cell magic, where text needs no quoting. Multi-command batches stay quiet because `%%exhash` expresses one command per cell.

In [ ]:
def _cmds(c):
    "Top-level command-tuple nodes of an exhash/file_exhash/cell_exhash call"
    if _callee(c) == 'exhash':
        a = c.args[1] if len(c.args) > 1 else None
        return a.elts if isinstance(a, (ast.List, ast.Tuple)) and any(isinstance(e, ast.Tuple) for e in a.elts) else []
    return c.args[2 if _callee(c) == 'cell_exhash' else 1:]

def _tuple_payload(tree, src):
    "A lone constant a/i/c payload that is long or contains quotes/backslashes belongs in a %%exhash cell; multi-command calls are exempt, since the one-command magic can't express them atomically"
    for c in _calls(tree):
        if _callee(c) not in ('exhash','file_exhash','cell_exhash'): continue
        cmds = _cmds(c)
        if len(cmds) != 1 or not isinstance(cmds[0], ast.Tuple): continue
        n = cmds[0]
        if len(n.elts) < 3: continue
        cmd,payload = n.elts[1],n.elts[2]
        if not (isinstance(cmd, ast.Constant) and cmd.value in ('a','i','c')): continue
        if not (isinstance(payload, ast.Constant) and isinstance(payload.value, str)): continue
        if len(payload.value) > 20 or any(ch in payload.value for ch in '\'"\\'): return True

RULES.append(Rule('tuple_payload', 'Apply a/i/c text with the %%exhash magic: it needs no quoting or escaping.', _tuple_payload))

Short quote-free payloads, `s` commands, variables, and multi-command calls stay quiet while lone long or escaped `a`/`i`/`c` tuples fire:

In [ ]:
hit = ("file_exhash(p, (a, 'c', 'a longer replacement line here'))", 'file_exhash(p, (a, "a", "it\'s"))',
    "cell_exhash(p, cid, (a, 'i', 'x = \\\\n'))", "cell_exhash(p, cid, (a, 'c', 'a longer replacement line here'))",
    "exhash(t, [(a, 'c', 'a longer replacement line here')])")
quiet = ("file_exhash(p, (a, 'c', 'metric'))", "file_exhash(p, (a, 's', 'longer than twenty chars ok'))",
    "file_exhash(p, (a, 'c', body))", "file_exhash(p, (a, 'd'), (b, 'c', 'a longer replacement line here'))",
    "cell_exhash(p, cid, (a, 's', 'x', 'y'), (b, 'i', 'it\'s a long insertion here'))")
assert all(fires(s, 'tuple_payload') for s in hit) and not any(fires(s, 'tuple_payload') for s in quiet)

Substitution (`s`) commands also check for a literal two-character `\n` in the replacement string (which stays literal instead of inserting a newline) and for replacements over 120 characters.

In [ ]:
def _s_repls(tree):
    "String-constant replacement fields of s-commands in exhash calls"
    for c in _calls(tree):
        if _callee(c) not in ('exhash','file_exhash','cell_exhash'): continue
        for n in ast.walk(c):
            if not (isinstance(n, ast.Tuple) and len(n.elts) >= 4): continue
            if not (isinstance(n.elts[1], ast.Constant) and n.elts[1].value == 's'): continue
            if isinstance(n.elts[3], ast.Constant) and isinstance(n.elts[3].value, str): yield n.elts[3].value

def _s_newline(tree, src): return any('\\n' in r for r in _s_repls(tree))
def _s_long(tree, src): return any(len(r) > 120 for r in _s_repls(tree))

RULES.append(Rule('s_newline', r'A 2-char \n in an s-replacement stays literal text: use a real newline in the string.', _s_newline))
RULES.append(Rule('s_long', 'Use a c command (%%exhash <addr> c) for an s-replacement over 120 chars, except when the line is much longer still.', _s_long))

Real newlines and pattern-side `\n` pass `s_newline`, and `s_long` triggers above 120 characters:

In [ ]:
assert fires(r"file_exhash(p, ('1|aa|', 's', 'x', r'a\nb'))", 's_newline')
assert not fires(r"file_exhash(p, ('1|aa|', 's', 'x', 'a\nb'))", 's_newline')
assert not fires(r"file_exhash(p, ('1|aa|', 's', r'x\n', 'y'))", 's_newline')
assert fires(f"file_exhash(p, ('1|aa|', 's', 'x', {'y'*121!r}))", 's_long')
assert not fires(f"file_exhash(p, ('1|aa|', 's', 'x', {'y'*120!r}))", 's_long')

## Tool output and magics

Toolkit views and search results render directly as their own repr; splitting or joining them discards formatting and line addresses. `_run_magic` checks the raw cell text (`raw=True`) because IPython transforms every legitimate `%magic` line into a `run_line_magic` call.

In [ ]:
_TOOLING = {'lnhashview','lnhashview_file','lnhashview_cell','lnhashview_cells','rg','nbrg','fd',
    'find_msgs','summary_dlg','view_dlg','view_msg','view_msgs','view_file','view_cell','doc','info_md'}

def _postproc(tree, src):
    for n in ast.walk(tree):
        if isinstance(n, ast.Attribute) and isinstance(n.value, ast.Call) and _callee(n.value) in _TOOLING \
           and n.attr in ('splitlines','split','join'): return True
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == 'join' \
           and isinstance(n.func.value, ast.Constant) \
           and any(_callee(c) in _TOOLING for a in n.args for c in _calls(a)): return True

def _run_magic(tree, src):
    # raw-source rule: in the transformed cell every magic becomes run_*_magic, so only literal raw uses count
    return 'run_line_magic' in src or 'run_cell_magic' in src

RULES.append(Rule('postproc', "Show tooling results bare; narrow with the tool's own parameters.", _postproc))
RULES.append(Rule('run_magic', 'Invoke magics directly with % syntax.', _run_magic, raw=True))

Bare toolkit views and `%magic` syntax pass, while joining/splitting results or calling `run_line_magic` explicitly fires:

In [ ]:
assert fires("'\\n'.join(lnhashview_file(p))", 'postproc')
assert fires("rg('x', p).splitlines()[:5]", 'postproc')
assert not fires('lnhashview_file(p)', 'postproc')
assert fires("get_ipython().run_line_magic('nbrun', 'abc')", 'run_magic')
assert not fires('%nbrun abc', 'run_magic')

## Blocking rules

Two rules stop the cell before it reaches the gateway: shell execution via `subprocess`, `os.system`, or `os.popen`, and mutating `sys.path`. Shell `!` escapes are left alone because BPL's factorial `!5` transforms into the same IPython call, and `io.popen` is left alone for Lua kernels.

In [ ]:
def _shell_escape(tree, src):
    for n in ast.walk(tree):
        if isinstance(n, ast.Import) and any(a.name.split('.')[0] == 'subprocess' for a in n.names): return True
        if isinstance(n, ast.ImportFrom) and (n.module or '').split('.')[0] == 'subprocess': return True
        if (isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr in ('system','popen')
            and isinstance(n.func.value, ast.Name) and n.func.value.id == 'os'): return True

def _sys_path(tree, src):
    for c in _calls(tree):
        f = c.func
        if isinstance(f, ast.Attribute) and f.attr in ('insert','append') and isinstance(f.value, ast.Attribute) \
           and f.value.attr == 'path' and isinstance(f.value.value, ast.Name) and f.value.value.id == 'sys': return True

RULES.append(Rule('shell_escape', 'Run shell commands with the Bash tool.', _shell_escape, block=True))
RULES.append(Rule('sys_path', 'Never modify sys.path; stop and ask the user.', _sys_path, block=True))

Python shell imports/calls and `sys.path` mutations block, while BPL `!5` and Lua `io.popen` stay quiet:

In [ ]:
shells = ('import subprocess', "os.system('ls')", "os.popen('ls')", 'import subprocess.x', 'from subprocess.x import y')
assert all(fires(s, 'shell_escape') for s in shells)
assert not fires('!5', 'shell_escape') and not fires("io.popen('ls')", 'shell_escape')
assert fires("sys.path.insert(0, 'x')", 'sys_path') and fires("sys.path.append('x')", 'sys_path')
test_eq([r.name for r in RULES if r.block], ['shell_escape', 'sys_path'])
[(r.name, 'blocking' if r.block else 'advisory') for r in RULES]